# 08 · Figuras comparativas — Artigo × Reprodução

- **Origem:** **notebook NOVO** (extensão) — gera figuras que colocam lado a lado os resultados do
  artigo (Leite et al., 2020) e os desta reprodução, no mesmo estilo, para comparação direta.
- **Faz:**
  - **Figura 2** (binário): matrizes de confusão dos 5 modelos, Artigo × Reproduzido.
  - **Figura 4** (multi-rótulo): matrizes de confusão dos 6 rótulos (mBERT), Artigo × Reproduzido.
  - **Curva de aprendizado** (Fig. R.3b): curvas reais sobrepostas (reprodução × artigo), a partir dos JSONs.
  - **Barras** de macro-F1 binário (Artigo × Reproduzido).
- **Entradas:** `results/binary_*.json`, `results/multilabel_mbert.json`, `results/learning_curve.json`
  (gerados pelo `03`) e `experiments/data/learning_curve.json` (curva dos autores, no repo).
- **Saídas:** `results/figures/comparison/compare_*.png`.
- **Números do artigo:** as matrizes/valores do artigo entram **hardcoded** abaixo como referência
  (lidos das Figuras 2–4 e Tabelas 7–11; ver `REPORT.md`).

> **Pré-requisito:** rode antes o `03_classification.ipynb` (gera os JSONs e `learning_curve.json`).


In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "ToLD-BR.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RESULTS   = ROOT / "reproduction" / "results"
FIGURES   = RESULTS / "figures"
OUT       = FIGURES / "comparison"
OUT.mkdir(parents=True, exist_ok=True)
print("RESULTS  :", RESULTS)
print("saída    :", OUT)


In [ ]:
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def load(name):
    return json.load(open(RESULTS / f"{name}.json", encoding="utf-8"))

def draw_cm(ax, cm, title, classes=("Não-tóxico", "Tóxico"), cmap="Blues"):
    cm = np.array(cm, dtype=int)
    vmax = cm.max() if cm.max() > 0 else 1
    ax.imshow(cm, cmap=cmap, vmin=0, vmax=vmax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(classes, fontsize=8)
    ax.set_yticklabels(classes, fontsize=8, rotation=90, va="center")
    ax.set_xlabel("Predito", fontsize=8); ax.set_ylabel("Real", fontsize=8)
    ax.set_title(title, fontsize=9)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                    color="white" if cm[i, j] > vmax * 0.55 else "black",
                    fontsize=12, fontweight="bold")


## Figura 2 — matrizes de confusão (binário)

`[[TN, FP], [FN, TP]]`. Baseline: o artigo usa **BoW+AutoML**; a reprodução usa **BoW+SVM**
(auto-sklearn é Linux-only).

In [3]:
# (rótulo, nome_artigo, cm_artigo, f1_artigo, nome_repro, json_repro, f1_repro)
bin_models = [
    ("BoW baseline",       "BoW+AutoML",       [[843, 285], [263, 709]], 0.74, "BoW+SVM",          "binary_baseline_bow_svm", 0.730),
    ("BR-BERT",            "BR-BERT",          [[902, 226], [267, 705]], 0.76, "BR-BERT",          "binary_br_bert",          0.765),
    ("M-BERT-BR",          "M-BERT-BR",        [[778, 350], [179, 793]], 0.75, "M-BERT-BR",        "binary_mbert_br",         0.766),
    ("M-BERT (transfer)",  "M-BERT(transfer)", [[837, 291], [207, 765]], 0.76, "M-BERT(transfer)", "binary_mbert_transfer",   0.752),
    ("M-BERT (zero-shot)", "M-BERT(zero-shot)",[[940, 188], [657, 315]], 0.56, "M-BERT(zero-shot)","binary_mbert_zeroshot",   0.544),
]
fig, axes = plt.subplots(5, 2, figsize=(7, 16))
for r, (lbl, p_name, p_cm, p_f1, o_name, o_json, o_f1) in enumerate(bin_models):
    o_cm = load(o_json)["confusion_matrix"]
    draw_cm(axes[r, 0], p_cm, f"Artigo — {p_name}\nmacro-F1 = {p_f1:.2f}")
    draw_cm(axes[r, 1], o_cm, f"Reproduzido — {o_name}\nmacro-F1 = {o_f1:.3f}")
fig.suptitle("Figura 2 — Matrizes de confusão (binário): Artigo × Reprodução", fontsize=12, y=0.997)
fig.tight_layout(rect=[0, 0, 1, 0.99])
fig.savefig(OUT / "compare_fig2_binary_cm.png", dpi=130, bbox_inches="tight")
plt.close(fig)
print("salvo:", OUT / "compare_fig2_binary_cm.png")

salvo: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\figures\comparison\compare_fig2_binary_cm.png


## Figura 4 — matrizes de confusão (multi-rótulo, mBERT)

Mostra o *trade-off* central: o artigo (esquerda) é conservador (poucos positivos preditos); a
reprodução (direita) prevê mais positivos nas classes frequentes (obscene, insult). Classes raras
(racism, xenophobia, misogyny) ficam ~tudo negativo em ambos.

In [4]:
# rótulo, cm_artigo, índice no per_label_confusion (homophobia,obscene,insult,racism,misogyny,xenophobia)
ml_labels = [
    ("LGBTQ+phobia", [[2072, 2], [25, 1]],   0),
    ("Obscene",      [[1430, 38], [427, 205]], 1),
    ("Insult",       [[1635, 41], [290, 134]], 2),
    ("Racism",       [[2089, 0], [11, 0]],    3),
    ("Misogyny",     [[2057, 0], [39, 4]],    4),
    ("Xenophobia",   [[2081, 0], [19, 0]],    5),
]
ml = load("multilabel_mbert")["per_label_confusion"]
fig, axes = plt.subplots(6, 2, figsize=(7, 18))
for r, (lbl, p_cm, idx) in enumerate(ml_labels):
    draw_cm(axes[r, 0], p_cm,   f"Artigo (mBERT) — {lbl}",      classes=("Neg.", "Pos."), cmap="Greens")
    draw_cm(axes[r, 1], ml[idx], f"Reproduzido (mBERT) — {lbl}", classes=("Neg.", "Pos."), cmap="Greens")
fig.suptitle("Figura 4 — Matrizes de confusão (multi-rótulo): Artigo × Reprodução", fontsize=12, y=0.997)
fig.tight_layout(rect=[0, 0, 1, 0.99])
fig.savefig(OUT / "compare_fig4_multilabel_cm.png", dpi=130, bbox_inches="tight")
plt.close(fig)
print("salvo:", OUT / "compare_fig4_multilabel_cm.png")

salvo: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\figures\comparison\compare_fig4_multilabel_cm.png


## Barras — macro-F1 binário (Artigo × Reprodução)

In [6]:
names    = [m[0] for m in bin_models]
paper_f1 = [m[3] for m in bin_models]
ours_f1  = [m[6] for m in bin_models]
xpos = np.arange(len(names)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(xpos - w/2, paper_f1, w, label="Artigo", color="#888888")
b2 = ax.bar(xpos + w/2, ours_f1,  w, label="Reproduzido", color="#1f77b4")
ax.set_xticks(xpos); ax.set_xticklabels(names, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("macro-F1"); ax.set_ylim(0.5, 0.82)
ax.set_title("Binário — macro-F1: Artigo × Reprodução"); ax.legend(); ax.grid(axis="y", alpha=0.3)
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.004,
            f"{b.get_height():.3f}".rstrip("0").rstrip("."), ha="center", fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "compare_macro_f1_bars.png", dpi=130, bbox_inches="tight")
plt.close(fig)
print("salvo:", OUT / "compare_macro_f1_bars.png")
print("\nTodas as figuras comparativas em:", OUT)

salvo: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\figures\comparison\compare_macro_f1_bars.png

Todas as figuras comparativas em: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\figures\comparison


## Figura R.3b — Curvas reais sobrepostas (artigo × reprodução)

Em vez de comparar a *nossa plotagem* contra a *imagem* do artigo (Figura R.3), esta figura sobrepõe
as **curvas reais**: o artigo vem do `experiments/data/learning_curve.json` (dados do próprio autor —
média das 3 repetições por fração, **incluindo os runs colapsados**, como o paper plota); a
reprodução vem de `results/learning_curve.json`. O eixo-x está em **% dos dados de treino**, o que
**neutraliza a diferença de base** (o artigo conta as frações sobre ~21.000 = dataset inteiro; a
reprodução sobre 18.900 = train+validation, mantendo o teste sempre disjunto).

In [7]:
# Figura R.3b — sobreposicao das curvas REAIS (artigo x reproducao), eixo-x em % dos dados de treino
paper_lc = json.load(open(ROOT / "experiments" / "data" / "learning_curve.json", encoding="utf-8"))
ours_lc = load("learning_curve")
xpct = list(range(10, 101, 10))
def _pmean(field): return [float(np.mean(paper_lc[str(k)][field])) for k in range(1, 11)]   # media das 3 reps
_fr = sorted(ours_lc, key=float)
def _omean(field): return [ours_lc[f][field] for f in _fr]

BLUE, ORANGE = "#1f77b4", "#ff7f0e"
fig, (axp, axn) = plt.subplots(1, 2, figsize=(14, 6))
for ax, suf, pr, ttl in [(axp, "pos", "positive", "Classe positiva (tóxica)"),
                         (axn, "neg", "negative", "Classe negativa (não-tóxica)")]:
    ax.plot(xpct, _omean(f"recall_{suf}"),    "o-",  color=BLUE,   label="recall — reprodução")
    ax.plot(xpct, _pmean(f"recall_{pr}"),     "x--", color=BLUE,   label="recall — artigo", alpha=0.8)
    ax.plot(xpct, _omean(f"precision_{suf}"), "o-",  color=ORANGE, label="precision — reprodução")
    ax.plot(xpct, _pmean(f"precision_{pr}"),  "x--", color=ORANGE, label="precision — artigo", alpha=0.8)
    ax.set_title(ttl, fontsize=12); ax.set_xlabel("% dos dados de treino"); ax.set_ylabel("score")
    ax.set_xticks(xpct); ax.set_ylim(0, 1.0); ax.grid(alpha=0.3); ax.legend(fontsize=9, ncol=2)
fig.suptitle("Curva de aprendizado — artigo × reprodução (curvas reais sobrepostas)\n"
             "eixo-x em % dos dados de treino (base: artigo 21.000 vs reprodução 18.900); "
             "sólida = reprodução, tracejada = artigo", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(OUT / "compare_fig3b_overlay_learning_curve.png", dpi=130, bbox_inches="tight")
plt.close(fig)
print("salvo:", OUT / "compare_fig3b_overlay_learning_curve.png")


salvo: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\figures\comparison\compare_fig3b_overlay_learning_curve.png
